# Layers & Modules

## 层和块

In [6]:
import torch
from torch import nn

# --- 块的基本结构 ---
# 自定义多层感知机
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.hidden = nn.Linear(20, 256)
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)
    
    def forward(self, X):
        return self.out(self.relu(self.hidden(X)))
    
# 使用 Sequential
class SeqMLP(nn.Module):
    def __init__(self):
        self.net = nn.Sequential(
            nn.Linear(20, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    
    def forward(self, X):
        return self.net(X)
    
# 使用嵌套
class BasicBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # 把 Linear 和 ReLU 封装成一个小整体
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU()
        )

    def forward(self, X):
        return self.net(X)

class MacroBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # 嵌套 3 个 Level 1 的 BasicBlock
        self.layer_stack = nn.Sequential(
            BasicBlock(dim, dim),
            BasicBlock(dim, dim),
            BasicBlock(dim, dim)
        )

    def forward(self, X):
        return self.layer_stack(X)
    
class FinalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64), # 输入层
            MacroBlock(64),    # 嵌套 Level 2 组件 A
            MacroBlock(64),    # 嵌套 Level 2 组件 B
            nn.Linear(64, 10)  # 输出层
        )

    def forward(self, X):
        return self.net(X)

# 实例化并观察结构
model = FinalNet()
print(model)


FinalNet(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=64, bias=True)
    (1): MacroBlock(
      (layer_stack): Sequential(
        (0): BasicBlock(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=64, bias=True)
            (1): ReLU()
          )
        )
        (1): BasicBlock(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=64, bias=True)
            (1): ReLU()
          )
        )
        (2): BasicBlock(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=64, bias=True)
            (1): ReLU()
          )
        )
      )
    )
    (2): MacroBlock(
      (layer_stack): Sequential(
        (0): BasicBlock(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=64, bias=True)
            (1): ReLU()
          )
        )
        (1): BasicBlock(
          (net): Sequential(
            (0): Linear(in_features=64, out_features=64, bias

## 参数管理
当定义好了一个复杂的嵌套块后，下一步就是如何精准地访问、初始化和共享这些参数，决定了是否能真正掌控模型

In [ ]:
# --- 访问特定层的参数 ---
# FinalNet -> 第三层 -> 第一个子块 -> 第一层的权重
print(model.net[2].layer_stack[0].net[0].weight)

# 一次性获取所有参数
for name, param in model.named_parameters():
    print(f"层名: {name}, 形状: {param.shape}")

# --- 初始化参数 ---
# 使用内置初始化器
def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean = 0, std = 0.01)
        nn.init .zeros_(m.bias)

model.apply(init_weights)

# 自定义初始化
def my_init(m):
    if type(m) == nn.Linear:
        m.weight.data.fill_(1)

# 参数共享
shared = nn.Linear(8, 8)
net = nn.Sequential(
    nn.Linear(16, 8),
    shared,          # 第一次使用
    nn.ReLU(),
    shared,          # 第二次使用，它们始终指向同一个内存空间
    nn.Linear(8, 1)
)

# 验证：修改其中一个，另一个也会变
print(net[1].weight == net[3].weight)

Parameter containing:
tensor([[ 0.0427,  0.0990,  0.0507,  ...,  0.0566, -0.0110,  0.0964],
        [ 0.0690,  0.0449,  0.0167,  ...,  0.1141,  0.0280,  0.0789],
        [ 0.0559, -0.0271,  0.1100,  ...,  0.0414,  0.1031,  0.0143],
        ...,
        [-0.0279,  0.0013, -0.0661,  ...,  0.0295, -0.0493, -0.0528],
        [ 0.0263,  0.0201,  0.0938,  ..., -0.0654, -0.0056,  0.0186],
        [ 0.0283, -0.0073, -0.0218,  ...,  0.0440, -0.1122,  0.1150]],
       requires_grad=True)
层名: net.0.weight, 形状: torch.Size([64, 20])
层名: net.0.bias, 形状: torch.Size([64])
层名: net.1.layer_stack.0.net.0.weight, 形状: torch.Size([64, 64])
层名: net.1.layer_stack.0.net.0.bias, 形状: torch.Size([64])
层名: net.1.layer_stack.1.net.0.weight, 形状: torch.Size([64, 64])
层名: net.1.layer_stack.1.net.0.bias, 形状: torch.Size([64])
层名: net.1.layer_stack.2.net.0.weight, 形状: torch.Size([64, 64])
层名: net.1.layer_stack.2.net.0.bias, 形状: torch.Size([64])
层名: net.2.layer_stack.0.net.0.weight, 形状: torch.Size([64, 64])
层名: net.2.laye

## 自定义层
有时会遇到或要自己发明一个现在在深度学习框架中还不存在的层，在这些情况下，必须构建自定义层

In [ ]:
# 不带参数的层
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()
    
# 带参数的自定义层
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        # 使用 nn.Parameter 包装张量，这样 PyTorch 才会自动追踪它的梯度
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return torch.relu(linear)

## 读写文件

In [ ]:
# --- 读写张量 ---
x = torch.arange(4)

# 存储
torch.save(x, 'x-file')

# 读取
x2 = torch.load('x-file')
print(x2)

# 存储为列表或字典
y = torch.zeros(4)
torch.save([x, y], 'x-files')
torch.save({'x': x, 'y': y}, 'mydict')

# --- 读写模型参数 ---
net = MLP()
# ... 假设这里经过了训练 ...
# 保存模型的参数字典
torch.save(net.state_dict(), 'mlp.params')

# 创建一个完全一样的“空壳”模型
clone = MLP()

# 将硬盘里的参数读入字典
params = torch.load('mlp.params')

# 将字典里的值加载到模型中
clone.load_state_dict(params)

# 验证（设置为评估模式）
clone.eval()

## GPU

In [ ]:
import torch

# 检查 CUDA 是否可用
print(torch.cuda.is_available()) 

# 查看有几个 GPU
print(torch.cuda.device_count())

# 获取当前设备的索引（如果有的话）
print(torch.cuda.current_device())

# --- 默认情况下，所有的张量都是创建在 CPU 上的 ---
# 在创建时指定设备
# 定义一个设备对象
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 直接在 GPU 上创建张量
x = torch.ones(2, 3, device=device)
print(x)

# 移动现有的张量（推荐）
y = torch.rand(2, 3)
y = y.to(device) # 将 CPU 上的 y 复制一份到 GPU

True
1
0
